# 🫀 ExtraCTOps Echo Report Extraction - Production Notebook

This notebook demonstrates **real-world clinical text extraction** using ExtraCTOps generators to extract structured echocardiogram data from CSV files using the comprehensive EchoReport Pydantic model.

## Features
- **Dual Generator Support**: Compare OpenAI and Ollama generators
- **Clinical-Grade Schema**: EchoReport model with comprehensive cardiac parameters
- **Batch Processing**: Process multiple echo reports efficiently
- **Production Ready**: Error handling, progress tracking, and result validation

## Prerequisites
1. CSV file with echo report text data
2. Valid OpenAI API key (for OpenAI generator)
3. Ollama running locally (for Ollama generator)
4. ExtraCTOps generators configured properly

## Use Cases
- Extract structured data from clinical echo reports
- Compare extraction quality between different LLM providers
- Create structured datasets for clinical research
- Validate model performance on real medical text

## 1. Import Required Libraries

Import all necessary libraries for data processing, LLM integration, and the ExtraCTOps framework.

In [ ]:
# Standard library imports
import os
import sys
import asyncio
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

# Data processing
import pandas as pd

# Add project root to Python path for imports
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# ExtraCTOps imports
from utils.ExtraCTOps_loops import ProcessingConfig, ExtraCTOpsProcessor, process_extraction_batch
from the_pydantics.EchoReport import EchoReport

print("✅ All libraries imported successfully!")
print(f"📁 Project root: {project_root}")
print(f"🐍 Python version: {sys.version}")
print(f"📊 Pandas version: {pd.__version__}")

## 2. Configuration and Setup

Configure the extraction parameters and validate the environment setup.

In [ ]:
# Configuration parameters
CONFIG = {
    # Input/Output settings
    "INPUT_FILE": "",  # Will be set by user input
    "TEXT_COLUMN": "echo_report_text",  # Column containing echo report text
    "UID_COLUMN": "patient_id",  # Column with unique patient identifier
    "OUTPUT_DIR": project_root / "exports" / "echo_extractions",
    
    # Processing settings
    "BATCH_SIZE": 5,  # Number of concurrent extractions
    "BACKUP_INTERVAL": 10,  # Save backup every N processed items
    "MAX_RETRIES": 2,  # Retry attempts for failed extractions
    
    # LLM settings
    "TEMPERATURE": 0.1,  # Low temperature for medical accuracy
    "MAX_TOKENS": 3000,  # Sufficient for complex echo reports
    
    # Generator models
    "OLLAMA_MODEL": "hermes3:8b-llama3.1-q8_0 ",  # or "llama3:70b" for larger model
    "OPENAI_MODEL": "gpt-4o",  # or "gpt-4o-mini" for faster/cheaper
    
    # System prompts for medical extraction
    "SYSTEM_MESSAGE": """You are a expert cardiologist and medical data extraction specialist. 
    Extract structured echocardiogram information from clinical reports with high accuracy. 
    Focus on cardiac anatomy, function, measurements, and pathology. 
    Return only valid JSON that matches the provided schema exactly.""",
    
    "PRE_PROMPT": """Analyze this echocardiogram report and extract all relevant cardiac information. 
    Include measurements with units, anatomical descriptions, functional assessments, and any abnormalities. 
    Be precise with medical terminology and numerical values. Return valid JSON only."""
}

# Create output directory
CONFIG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

print("✅ Configuration loaded:")
for key, value in CONFIG.items():
    if key not in ["SYSTEM_MESSAGE", "PRE_PROMPT"]:  # Skip long text
        print(f"   {key}: {value}")

print(f"\n📁 Output directory: {CONFIG['OUTPUT_DIR']}")
print(f"🫀 Using EchoReport model with {len(EchoReport.model_fields)} main sections")

## 3. Create Sample Echo Report Data (Optional)

Create sample echo report data for demonstration purposes, or skip to load your own CSV file.

In [ ]:
# Create sample echo report data
def create_sample_echo_data():
    """Create realistic sample echo report data for testing"""
    sample_reports = [
        {
            "patient_id": "ECHO_001",
            "study_date": "2024-01-15",
            "echo_report_text": """
ECHOCARDIOGRAM REPORT

Patient: 45-year-old male
Indication: Chest pain, rule out cardiac cause

FINDINGS:
Left Ventricle: The left ventricle is normal in size. Left ventricular systolic function is normal with an estimated ejection fraction of 65%. No regional wall motion abnormalities. LV diastolic volume 110 mL, systolic volume 38 mL.

Right Ventricle: The right ventricle is normal in size and systolic function.

Atria: The left atrium is mildly dilated. Right atrium is normal in size.

Valves: 
- Mitral valve is structurally normal with mild regurgitation
- Tricuspid valve shows mild regurgitation with estimated PA pressure 25 mmHg
- Aortic valve is structurally normal, trileaflet, no stenosis or regurgitation
- Pulmonary valve is normal with trivial regurgitation

Aorta: Aortic root measures 32 mm, ascending aorta 28 mm. Left aortic arch.

No pericardial effusion. No evidence of pulmonary hypertension.

IMPRESSION: Normal left ventricular size and systolic function. Mild left atrial dilation. Mild mitral and tricuspid regurgitation.
            """
        },
        {
            "patient_id": "ECHO_002", 
            "study_date": "2024-01-16",
            "echo_report_text": """
ECHOCARDIOGRAM REPORT

Patient: 62-year-old female
Indication: Hypertension, assessment of cardiac function

FINDINGS:
Left Ventricle: Moderate left ventricular hypertrophy. Estimated ejection fraction 45%, mildly depressed systolic function. LV diastolic volume 145 mL, systolic volume 80 mL.

Right Ventricle: Normal right ventricular size and function.

Atria: Both atria are moderately dilated. Left atrial volume indexed 38 mL/m².

Valves:
- Mitral valve shows mild stenosis and moderate regurgitation
- Aortic valve has mild stenosis with peak gradient 35 mmHg, mean gradient 20 mmHg
- Tricuspid regurgitation is moderate with elevated PA pressure 45 mmHg
- Pulmonary valve is normal

Great Vessels: Aortic root 35 mm, ascending aorta 40 mm.

Moderate pulmonary hypertension present with interventricular septal flattening in systole.

IMPRESSION: Moderate LV hypertrophy with mild systolic dysfunction. Moderate pulmonary hypertension. Mild aortic stenosis, moderate mitral regurgitation.
            """
        },
        {
            "patient_id": "ECHO_003",
            "study_date": "2024-01-17", 
            "echo_report_text": """
PEDIATRIC ECHOCARDIOGRAM REPORT

Patient: 8-year-old male
Indication: Heart murmur

FINDINGS:
Left Ventricle: Normal left ventricular size and systolic function, EF 65%.

Right Ventricle: Mild right ventricular dilation with normal systolic function.

Atria: Normal atrial sizes.

Septal Defects: 
- Small perimembranous ventricular septal defect, 4 mm, with left-to-right shunt
- Peak gradient across VSD 65 mmHg
- No atrial septal defect

Valves: All valves are structurally normal and competent.

Great Vessels: Normal aortic arch, no coarctation. Patent ductus arteriosus is absent.

Mild elevation of right heart pressures secondary to VSD.

IMPRESSION: Small perimembranous VSD with left-to-right shunt. Mild RV dilation. Normal valves and great vessels.
            """
        }
    ]
    
    df = pd.DataFrame(sample_reports)
    sample_file = project_root / "data" / "sample_echo_reports.csv"
    sample_file.parent.mkdir(exist_ok=True)
    df.to_csv(sample_file, index=False)
    
    print(f"✅ Created sample data: {sample_file}")
    print(f"📊 Sample data shape: {df.shape}")
    print(f"📋 Columns: {list(df.columns)}")
    
    return str(sample_file)

# Uncomment to create sample data
sample_file_path = create_sample_echo_data()
print(f"\n🎯 Sample file ready at: {sample_file_path}")
print("   You can use this file or provide your own CSV file path below.")

## 4. Load CSV Data

Load your echo report data from a CSV file. The CSV should contain columns for patient ID and echo report text.

In [ ]:
# Data loading and validation
def load_and_validate_data(file_path: str) -> pd.DataFrame:
    """Load CSV data and validate required columns"""
    try:
        df = pd.read_csv(file_path)
        print(f"✅ Loaded data: {df.shape}")
        print(f"📋 Columns: {list(df.columns)}")
        
        # Validate required columns
        required_cols = [CONFIG["UID_COLUMN"], CONFIG["TEXT_COLUMN"]]
        missing_cols = [col for col in required_cols if col not in df.columns]
        
        if missing_cols:
            print(f"❌ Missing required columns: {missing_cols}")
            print(f"   Available columns: {list(df.columns)}")
            print("   Please update CONFIG with correct column names or rename columns in CSV")
            return None
        
        # Show data preview
        print(f"\n📊 Data Preview:")
        print(df[required_cols].head())
        
        # Check for empty text fields
        empty_text = df[CONFIG["TEXT_COLUMN"]].isna().sum()
        if empty_text > 0:
            print(f"⚠️  Warning: {empty_text} rows have empty text fields")
            
        return df
        
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None

# Set input file path
# Option 1: Use sample data
input_file_path = sample_file_path

# Option 2: Specify your own file path (uncomment and modify)
# input_file_path = "/path/to/your/echo_reports.csv"

# Option 3: Interactive input (uncomment for interactive use)
# input_file_path = input("Enter path to your CSV file: ").strip()

# Update configuration
CONFIG["INPUT_FILE"] = input_file_path

# Load data
df = load_and_validate_data(CONFIG["INPUT_FILE"])

if df is not None:
    print(f"\n🎯 Ready to process {len(df)} echo reports")
    print(f"📁 Input file: {CONFIG['INPUT_FILE']}")
else:
    print("❌ Data loading failed. Please check file path and column names.")

## 5. Extract Using Ollama Generator

Process echo reports using the Ollama generator for local LLM extraction.

In [ ]:
# Extract using Ollama generator
async def run_ollama_extraction():
    """Run extraction using Ollama generator"""
    if df is None:
        print("❌ No data loaded. Please load data first.")
        return None
    
    print("🦙 Starting Ollama extraction...")
    print(f"   Model: {CONFIG['OLLAMA_MODEL']}")
    print(f"   Batch size: {CONFIG['BATCH_SIZE']}")
    print(f"   Temperature: {CONFIG['TEMPERATURE']}")
    
    try:
        # Configure Ollama extraction
        ollama_config = ProcessingConfig(
            input_file=CONFIG["INPUT_FILE"],
            text_column=CONFIG["TEXT_COLUMN"],
            uid_column=CONFIG["UID_COLUMN"],
            pydantic_model=EchoReport,
            generator_type="ollama",
            model_name=CONFIG["OLLAMA_MODEL"],
            experiment_label="ollama_echo_extraction",
            batch_size=CONFIG["BATCH_SIZE"],
            backup_interval=CONFIG["BACKUP_INTERVAL"],
            max_retries=CONFIG["MAX_RETRIES"],
            temperature=CONFIG["TEMPERATURE"],
            max_tokens=CONFIG["MAX_TOKENS"],
            system_message=CONFIG["SYSTEM_MESSAGE"],
            pre_prompt=CONFIG["PRE_PROMPT"],
            output_file=str(CONFIG["OUTPUT_DIR"] / f"echo_ollama_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
        )
        
        # Run extraction
        processor = ExtraCTOpsProcessor(ollama_config)
        await processor.process_batch(open_file=False)
        
        # Show results
        successful = len([r for r in processor.results if r.success])
        failed = len([r for r in processor.results if not r.success])
        avg_time = sum(r.execution_time for r in processor.results) / len(processor.results) if processor.results else 0
        
        print(f"\n✅ Ollama extraction completed!")
        print(f"   ✓ Successful: {successful}")
        print(f"   ✗ Failed: {failed}")
        print(f"   ⏱️ Average time: {avg_time:.2f}s per report")
        print(f"   📁 Output: {ollama_config.output_file}")
        
        return processor, ollama_config
        
    except Exception as e:
        print(f"❌ Ollama extraction failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# Run Ollama extraction
ollama_processor, ollama_config = await run_ollama_extraction()

## 6. Extract Using OpenAI Generator

Process the same echo reports using OpenAI's API for comparison with Ollama results.

In [ ]:
# Extract using OpenAI generator
async def run_openai_extraction():
    """Run extraction using OpenAI generator"""
    if df is None:
        print("❌ No data loaded. Please load data first.")
        return None
    
    print("🤖 Starting OpenAI extraction...")
    print(f"   Model: {CONFIG['OPENAI_MODEL']}")
    print(f"   Batch size: {CONFIG['BATCH_SIZE']}")
    print(f"   Temperature: {CONFIG['TEMPERATURE']}")
    
    # Check for OpenAI API key
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("⚠️  Warning: OPENAI_API_KEY not found in environment variables")
        print("   Set your API key with: export OPENAI_API_KEY='your-key-here'")
        print("   Or add it to your .env file")
        return None, None
    
    try:
        # Configure OpenAI extraction
        openai_config = ProcessingConfig(
            input_file=CONFIG["INPUT_FILE"],
            text_column=CONFIG["TEXT_COLUMN"],
            uid_column=CONFIG["UID_COLUMN"],
            pydantic_model=EchoReport,
            generator_type="openai",
            model_name=CONFIG["OPENAI_MODEL"],
            experiment_label="openai_echo_extraction",
            batch_size=CONFIG["BATCH_SIZE"],
            backup_interval=CONFIG["BACKUP_INTERVAL"],
            max_retries=CONFIG["MAX_RETRIES"],
            temperature=CONFIG["TEMPERATURE"],
            max_tokens=CONFIG["MAX_TOKENS"],
            system_message=CONFIG["SYSTEM_MESSAGE"],
            pre_prompt=CONFIG["PRE_PROMPT"],
            output_file=str(CONFIG["OUTPUT_DIR"] / f"echo_openai_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
        )
        
        # Run extraction
        processor = ExtraCTOpsProcessor(openai_config)
        await processor.process_batch(open_file=False)
        
        # Show results
        successful = len([r for r in processor.results if r.success])
        failed = len([r for r in processor.results if not r.success])
        avg_time = sum(r.execution_time for r in processor.results) / len(processor.results) if processor.results else 0
        
        print(f"\n✅ OpenAI extraction completed!")
        print(f"   ✓ Successful: {successful}")
        print(f"   ✗ Failed: {failed}")
        print(f"   ⏱️ Average time: {avg_time:.2f}s per report")
        print(f"   📁 Output: {openai_config.output_file}")
        
        return processor, openai_config
        
    except Exception as e:
        print(f"❌ OpenAI extraction failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# Run OpenAI extraction
openai_processor, openai_config = await run_openai_extraction()

## 7. Analyze and Compare Results

Compare the extraction results between Ollama and OpenAI generators.

In [ ]:
# Analyze and compare results
def analyze_extraction_results():
    """Analyze and compare extraction results from both generators"""
    print("📊 EXTRACTION RESULTS ANALYSIS")
    print("=" * 50)
    
    results_summary = {}
    
    # Analyze Ollama results
    if ollama_processor and ollama_processor.results:
        ollama_results = ollama_processor.results
        ollama_successful = len([r for r in ollama_results if r.success])
        ollama_failed = len([r for r in ollama_results if not r.success])
        ollama_avg_time = sum(r.execution_time for r in ollama_results) / len(ollama_results)
        
        results_summary["Ollama"] = {
            "total": len(ollama_results),
            "successful": ollama_successful,
            "failed": ollama_failed,
            "success_rate": (ollama_successful / len(ollama_results)) * 100,
            "avg_time": ollama_avg_time,
            "output_file": ollama_config.output_file if ollama_config else "N/A"
        }
        
        print(f"🦙 OLLAMA RESULTS:")
        print(f"   Total reports: {len(ollama_results)}")
        print(f"   Successful: {ollama_successful} ({(ollama_successful/len(ollama_results))*100:.1f}%)")
        print(f"   Failed: {ollama_failed}")
        print(f"   Avg time: {ollama_avg_time:.2f}s per report")
        print(f"   Output: {ollama_config.output_file if ollama_config else 'N/A'}")
        
        # Show sample successful extraction
        successful_results = [r for r in ollama_results if r.success and r.extracted_data]
        if successful_results:
            sample = successful_results[0]
            print(f"   Sample extraction keys: {list(sample.extracted_data.keys())[:5]}...")
    else:
        print("🦙 OLLAMA: No results available")
    
    print()
    
    # Analyze OpenAI results  
    if openai_processor and openai_processor.results:
        openai_results = openai_processor.results
        openai_successful = len([r for r in openai_results if r.success])
        openai_failed = len([r for r in openai_results if not r.success])
        openai_avg_time = sum(r.execution_time for r in openai_results) / len(openai_results)
        
        results_summary["OpenAI"] = {
            "total": len(openai_results),
            "successful": openai_successful,
            "failed": openai_failed,
            "success_rate": (openai_successful / len(openai_results)) * 100,
            "avg_time": openai_avg_time,
            "output_file": openai_config.output_file if openai_config else "N/A"
        }
        
        print(f"🤖 OPENAI RESULTS:")
        print(f"   Total reports: {len(openai_results)}")
        print(f"   Successful: {openai_successful} ({(openai_successful/len(openai_results))*100:.1f}%)")
        print(f"   Failed: {openai_failed}")
        print(f"   Avg time: {openai_avg_time:.2f}s per report")
        print(f"   Output: {openai_config.output_file if openai_config else 'N/A'}")
        
        # Show sample successful extraction
        successful_results = [r for r in openai_results if r.success and r.extracted_data]
        if successful_results:
            sample = successful_results[0]
            print(f"   Sample extraction keys: {list(sample.extracted_data.keys())[:5]}...")
    else:
        print("🤖 OPENAI: No results available")
    
    # Comparison
    if len(results_summary) >= 2:
        print("\n🔍 COMPARISON:")
        ollama_success = results_summary["Ollama"]["success_rate"]
        openai_success = results_summary["OpenAI"]["success_rate"]
        ollama_time = results_summary["Ollama"]["avg_time"]
        openai_time = results_summary["OpenAI"]["avg_time"]
        
        print(f"   Success Rate: Ollama {ollama_success:.1f}% vs OpenAI {openai_success:.1f}%")
        print(f"   Speed: Ollama {ollama_time:.2f}s vs OpenAI {openai_time:.2f}s per report")
        
        if ollama_success > openai_success:
            print("   🏆 Ollama has higher success rate")
        elif openai_success > ollama_success:
            print("   🏆 OpenAI has higher success rate")
        else:
            print("   🤝 Both generators have equal success rates")
            
        if ollama_time < openai_time:
            print("   ⚡ Ollama is faster")
        elif openai_time < ollama_time:
            print("   ⚡ OpenAI is faster")
        else:
            print("   ⚖️ Both generators have similar speeds")
    
    print(f"\n📁 All output files saved to: {CONFIG['OUTPUT_DIR']}")
    
    return results_summary

# Run analysis
results_summary = analyze_extraction_results()

## 8. Inspect Extracted Data

Examine the extracted EchoReport data structure and validate the results.

In [ ]:
# Inspect extracted data
def inspect_extracted_data():
    """Inspect and display sample extracted data"""
    print("🔍 EXTRACTED DATA INSPECTION")
    print("=" * 50)
    
    # Function to show sample extraction
    def show_sample_extraction(processor, generator_name):
        if not processor or not processor.results:
            print(f"❌ No {generator_name} results available")
            return
            
        successful_results = [r for r in processor.results if r.success and r.extracted_data]
        if not successful_results:
            print(f"❌ No successful {generator_name} extractions")
            return
            
        sample = successful_results[0]
        print(f"\n🔍 {generator_name.upper()} SAMPLE EXTRACTION:")
        print(f"   Patient ID: {sample.uid}")
        print(f"   Execution Time: {sample.execution_time:.2f}s")
        print(f"   Success: {sample.success}")
        
        if sample.extracted_data:
            print(f"   📊 Extracted {len(sample.extracted_data)} fields:")
            
            # Group fields by category for better display
            field_groups = {}
            for field_name, value in sample.extracted_data.items():
                if '_' in field_name:
                    category = field_name.split('_')[0]
                else:
                    category = 'other'
                    
                if category not in field_groups:
                    field_groups[category] = []
                field_groups[category].append((field_name, value))
            
            # Display grouped fields
            for category, fields in field_groups.items():
                if len(fields) <= 5:  # Show all if few fields
                    for field_name, value in fields:
                        print(f"      {field_name}: {value}")
                else:  # Show sample if many fields
                    print(f"      {category.upper()} ({len(fields)} fields):")
                    for field_name, value in fields[:3]:
                        print(f"        {field_name}: {value}")
                    print(f"        ... and {len(fields)-3} more")
                print()
        
        # Show raw response sample (first 200 chars)
        if sample.raw_response:
            print(f"   📝 Raw Response Sample: {sample.raw_response[:200]}...")
    
    # Show samples from both generators
    show_sample_extraction(ollama_processor, "Ollama")
    show_sample_extraction(openai_processor, "OpenAI")
    
    # Load and display output files
    print(f"\n📁 OUTPUT FILES:")
    
    def load_and_preview_output(config, generator_name):
        if not config or not config.output_file:
            print(f"❌ No {generator_name} output file")
            return
            
        try:
            output_df = pd.read_excel(config.output_file)
            print(f"\n📊 {generator_name.upper()} OUTPUT FILE:")
            print(f"   File: {config.output_file}")
            print(f"   Shape: {output_df.shape}")
            print(f"   Columns: {len(output_df.columns)}")
            
            # Show extraction status
            status_col = f"{config.experiment_label}_status"
            if status_col in output_df.columns:
                status_counts = output_df[status_col].value_counts()
                print(f"   Status distribution:")
                for status, count in status_counts.items():
                    print(f"      {status}: {count}")
            
            # Show sample extracted columns
            extraction_cols = [col for col in output_df.columns if config.experiment_label in col and col != status_col]
            if extraction_cols:
                print(f"   Extracted fields: {len(extraction_cols)}")
                print(f"   Sample fields: {extraction_cols[:5]}")
                
        except Exception as e:
            print(f"❌ Error loading {generator_name} output: {e}")
    
    load_and_preview_output(ollama_config, "Ollama")
    load_and_preview_output(openai_config, "OpenAI")

# Run inspection
inspect_extracted_data()

## 9. Save and Export Results

Create final summary reports and export results in multiple formats.

In [ ]:
# Save and export final results
def create_final_summary_report():
    """Create comprehensive summary report of all extractions"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Create summary report
    summary_report = {
        "extraction_session": {
            "timestamp": timestamp,
            "input_file": CONFIG["INPUT_FILE"],
            "total_reports": len(df) if df is not None else 0,
            "configuration": {
                "batch_size": CONFIG["BATCH_SIZE"],
                "temperature": CONFIG["TEMPERATURE"],
                "max_tokens": CONFIG["MAX_TOKENS"],
                "ollama_model": CONFIG["OLLAMA_MODEL"],
                "openai_model": CONFIG["OPENAI_MODEL"]
            }
        },
        "generator_results": results_summary,
        "pydantic_model": {
            "name": "EchoReport",
            "fields": list(EchoReport.__fields__.keys()),
            "total_possible_fields": len(EchoReport.__fields__)
        }
    }
    
    # Save summary as JSON
    summary_file = CONFIG["OUTPUT_DIR"] / f"extraction_summary_{timestamp}.json"
    with open(summary_file, 'w') as f:
        json.dump(summary_report, f, indent=2, default=str)
    
    print(f"✅ Summary report saved: {summary_file}")
    
    # Create comparison CSV if both generators ran
    if ollama_config and openai_config:
        try:
            # Load both output files
            ollama_df = pd.read_excel(ollama_config.output_file)
            openai_df = pd.read_excel(openai_config.output_file)
            
            # Create comparison DataFrame
            comparison_data = []
            for _, row in df.iterrows():
                uid = row[CONFIG["UID_COLUMN"]]
                
                # Get Ollama results
                ollama_row = ollama_df[ollama_df[CONFIG["UID_COLUMN"]] == uid]
                ollama_status = ollama_row.iloc[0][f"{ollama_config.experiment_label}_status"] if not ollama_row.empty else "Not found"
                
                # Get OpenAI results  
                openai_row = openai_df[openai_df[CONFIG["UID_COLUMN"]] == uid]
                openai_status = openai_row.iloc[0][f"{openai_config.experiment_label}_status"] if not openai_row.empty else "Not found"
                
                comparison_data.append({
                    "patient_id": uid,
                    "ollama_status": ollama_status,
                    "openai_status": openai_status,
                    "both_successful": ollama_status == "EXTRACTED" and openai_status == "EXTRACTED",
                    "any_successful": ollama_status == "EXTRACTED" or openai_status == "EXTRACTED"
                })
            
            comparison_df = pd.DataFrame(comparison_data)
            comparison_file = CONFIG["OUTPUT_DIR"] / f"generator_comparison_{timestamp}.csv"
            comparison_df.to_csv(comparison_file, index=False)
            
            print(f"✅ Comparison report saved: {comparison_file}")
            
            # Show comparison stats
            both_success = comparison_df["both_successful"].sum()
            any_success = comparison_df["any_successful"].sum()
            total = len(comparison_df)
            
            print(f"\n📊 COMPARISON SUMMARY:")
            print(f"   Both successful: {both_success}/{total} ({(both_success/total)*100:.1f}%)")
            print(f"   Any successful: {any_success}/{total} ({(any_success/total)*100:.1f}%)")
            
        except Exception as e:
            print(f"❌ Error creating comparison: {e}")
    
    return summary_report

def export_structured_data():
    """Export structured data in JSON format for downstream analysis"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Function to extract structured data from processor results
    def extract_structured_results(processor, generator_name):
        if not processor or not processor.results:
            return []
            
        structured_data = []
        for result in processor.results:
            if result.success and result.extracted_data:
                # Reconstruct EchoReport structure from flattened data
                echo_data = {
                    "patient_id": result.uid,
                    "generator": generator_name,
                    "execution_time": result.execution_time,
                    "extracted_fields": result.extracted_data,
                    "extraction_success": True
                }
            else:
                echo_data = {
                    "patient_id": result.uid,
                    "generator": generator_name,
                    "execution_time": result.execution_time,
                    "error": result.error,
                    "extraction_success": False
                }
            structured_data.append(echo_data)
        
        return structured_data
    
    # Collect all structured data
    all_structured_data = []
    
    if ollama_processor:
        ollama_data = extract_structured_results(ollama_processor, "ollama")
        all_structured_data.extend(ollama_data)
    
    if openai_processor:
        openai_data = extract_structured_results(openai_processor, "openai")
        all_structured_data.extend(openai_data)
    
    if all_structured_data:
        structured_file = CONFIG["OUTPUT_DIR"] / f"structured_echo_data_{timestamp}.json"
        with open(structured_file, 'w') as f:
            json.dump(all_structured_data, f, indent=2, default=str)
        
        print(f"✅ Structured data exported: {structured_file}")
        print(f"   Total records: {len(all_structured_data)}")
    
    return all_structured_data

# Create final reports
print("📋 Creating final summary reports...")
summary_report = create_final_summary_report()

print("\n📦 Exporting structured data...")
structured_data = export_structured_data()

print(f"\n🎉 EXTRACTION SESSION COMPLETE!")
print(f"📁 All files saved to: {CONFIG['OUTPUT_DIR']}")
print(f"📊 Total files generated: {len(list(CONFIG['OUTPUT_DIR'].glob('*')))}")

# List all generated files
print(f"\n📋 Generated Files:")
for file_path in sorted(CONFIG["OUTPUT_DIR"].glob("*")):
    file_size = file_path.stat().st_size / 1024  # KB
    print(f"   📄 {file_path.name} ({file_size:.1f} KB)")

print(f"\n✨ Ready for clinical research and analysis!")

## 🎯 Conclusion and Next Steps

This notebook successfully demonstrated **production-grade clinical text extraction** using ExtraCTOps generators with the comprehensive EchoReport Pydantic model.

### ✅ What We Accomplished

1. **Dual Generator Comparison**: Processed echo reports with both Ollama (local) and OpenAI (API) generators
2. **Clinical-Grade Schema**: Used the comprehensive EchoReport model with nested cardiac structures
3. **Batch Processing**: Efficiently processed multiple reports with error handling and progress tracking
4. **Quality Assessment**: Compared extraction quality, speed, and success rates between generators
5. **Comprehensive Output**: Generated Excel files, JSON summaries, and comparison reports

### 📊 Key Outputs

- **Excel Files**: Original data + extracted fields + metadata for each generator
- **JSON Summaries**: Processing statistics and configuration details
- **Comparison Reports**: Side-by-side generator performance analysis
- **Structured Data**: Clean JSON exports for downstream analysis

### 🚀 Next Steps

1. **Scale Up**: Process larger datasets (hundreds/thousands of echo reports)
2. **Quality Validation**: Compare extracted data with ground truth annotations
3. **Model Optimization**: Fine-tune prompts and parameters for higher accuracy
4. **Integration**: Connect to clinical databases or FHIR systems
5. **Evaluation**: Use ExtraCTOps evaluators to assess extraction quality
6. **Research**: Analyze extracted data for clinical insights and patterns

### 🔧 Production Tips

- **API Keys**: Ensure OpenAI API key is properly configured for cloud processing
- **Local Models**: Consider larger Ollama models (llama3:70b) for complex clinical text
- **Batch Sizes**: Adjust based on system resources and API rate limits
- **Backup Strategy**: Regular backups prevent data loss during long processing sessions
- **Error Monitoring**: Review failed extractions to improve prompts and handling

### 📚 Further Reading

- ExtraCTOps Documentation: `/utils/ExtraCTOps_loops/README.md`
- Generator Documentation: `/generators/Readme.md`
- EchoReport Schema: `/the_pydantics/EchoReport.py`
- Configuration Guide: `/config/README.md`

---

**Ready for clinical research and real-world deployment! 🫀✨**